# 03. Functions, Hnadling NULLs and Sorting

- Date Functions
- String Function
- Working with NULLs
- Type Conversion
- Deduplicaing and Sorting

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime, time

data = [
    Row(id=1, name="Alice", days_active=120, join_timestamp=datetime(2024, 1, 15, 9, 30, 0), join_time=time(9, 30, 0)),
    Row(id=2, name="Bob", days_active=200, join_timestamp=datetime(2023, 12, 5, 14, 0, 0), join_time=time(14, 0, 0)),
    Row(id=3, name="Charlie", days_active=30, join_timestamp=datetime(2025, 6, 30, 8, 15, 0), join_time=time(8, 15, 0)),
    Row(id=4, name="Diana", days_active=60, join_timestamp=datetime(2026, 2, 28, 16, 45, 0), join_time=time(16, 45, 0))
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("days_active", IntegerType(), True),
    StructField("join_timestamp", TimestampType(), True),
    StructField("join_time", StringType(), True)  # Spark does not have a native TimeType, so use StringType
])

# Convert time objects to string for join_time
data = [
    Row(
        id=row.id,
        name=row.name,
        days_active=row.days_active,
        join_timestamp=row.join_timestamp,
        join_time=row.join_time.strftime("%H:%M:%S")
    )
    for row in data
]

df = spark.createDataFrame(data, schema)
display(df)

## Date Functions

Covert to date

In [0]:
from pyspark.sql import functions as F


df2 = df.withColumn("join_date", F.to_date("join_timestamp")) \
        .withColumn("join_time_ts", F.to_timestamp(F.concat(F.lit("1970-01-01 "), F.col("join_time"))))\
        .withColumn("join_date_time_ts", F.to_timestamp(F.concat(F.col("join_date"),F.lit(" ") ,F.col("join_time"))))
display(df2)



In [0]:
df2.printSchema()

Add Current timestamp

In [0]:
demo_current = df2.select(
    "id",
    "name",
    F.current_date().alias("current_date"),
    F.current_timestamp().alias("current_timestamp")
)

display(demo_current)
demo_current.printSchema()

Extract year, month, day, hour, minute, second

In [0]:
demo_extract = df2.select(
    "id",
    "name",
    "join_timestamp",
    F.year("join_timestamp").alias("year"),
    F.month("join_timestamp").alias("month"),
    F.dayofmonth("join_timestamp").alias("day_of_month"),
    F.dayofweek("join_timestamp").alias("day_of_week"),
    F.dayofyear("join_timestamp").alias("day_of_year"),
    F.weekofyear("join_timestamp").alias("week_of_year"),
    F.hour("join_timestamp").alias("hour"),
    F.minute("join_timestamp").alias("minute"),
    F.second("join_timestamp").alias("second")
)

display(demo_extract)

Date formatting

- YYYY-MM-DD  -- Universal
- DD-MM-YYYY  - UK
- DD-MM-YYYY -- IND
- MM-DD-YYYY -- US

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_format.html


Read about Date dimension

In [0]:
demo_format = df2.select(
    "id",
    "name",
    "join_timestamp",
    F.date_format("join_timestamp", "yyyy-MM-dd").alias("date_format_1"),
    F.date_format("join_timestamp", "dd-MM-yyyy").alias("date_format_2"),
    F.date_format("join_timestamp", "EEEE").alias("day_name"),
    F.date_format("join_timestamp", "EEE").alias("day_short_name"),
    F.date_format("join_timestamp", "MMM").alias("month_short"),
    F.date_format("join_timestamp", "MMMM").alias("month_long"),
    F.date_format("join_timestamp", "HH:mm:ss").alias("time_only")
)

display(demo_format)

- date dimention

Date add and date subtract

In [0]:
demo_date_add_sub = df2.select(
    "id",
    "name",
    "join_date",
    F.date_add("join_date", 10).alias("plus_10_days"),
    F.date_sub("join_date", 10).alias("minus_10_days")
)

display(demo_date_add_sub)

Add months and find last day of month

In [0]:
demo_month_ops = df2.select(
    "id",
    "name",
    "join_date",
    F.add_months("join_date", 2).alias("plus_2_months"),
    F.last_day("join_date").alias("last_day_of_month"),
    F.trunc("join_date", "month").alias("month_start"),
    F.trunc("join_date", "year").alias("year_start")
)

display(demo_month_ops)

Date difference and months between

In [0]:
demo_diff = df2.select(
    "id",
    "name",
    "join_date",
    F.current_date().alias("today"),
    F.datediff(F.current_date(), F.col("join_date")).alias("days_since_join"),
    F.months_between(F.current_date(), F.col("join_date")).alias("months_since_join")
)

display(demo_diff)

Next day and quarter

In [0]:
demo_next_day = df2.select(
    "id",
    "name",
    "join_date",
    F.next_day("join_date", "Monday").alias("next_monday"),
    F.quarter("join_date").alias("quarter")
)

display(demo_next_day)

Date truncation

In [0]:
demo_date_trunc = df2.select(
    "id",
    "name",
    "join_timestamp",
    F.date_trunc("year", "join_timestamp").alias("trunc_year"),
    F.date_trunc("month", "join_timestamp").alias("trunc_month"),
    F.date_trunc("day", "join_timestamp").alias("trunc_day"),
    F.date_trunc("hour", "join_timestamp").alias("trunc_hour")
)

display(demo_date_trunc)

All at once

In [0]:
demo_datetime = df.withColumn("join_date", F.to_date("join_timestamp")) \
    .withColumn("join_time_ts", F.to_timestamp(F.concat(F.lit("1970-01-01 "), F.col("join_time")))) \
    .select(
        "id",
        "name",
        "days_active",
        "join_timestamp",
        "join_time",
        "join_date",
        F.year("join_timestamp").alias("year"),
        F.month("join_timestamp").alias("month"),
        F.dayofmonth("join_timestamp").alias("day_of_month"),
        F.dayofweek("join_timestamp").alias("day_of_week"),
        F.dayofyear("join_timestamp").alias("day_of_year"),
        F.weekofyear("join_timestamp").alias("week_of_year"),
        F.hour("join_timestamp").alias("ts_hour"),
        F.minute("join_timestamp").alias("ts_minute"),
        F.second("join_timestamp").alias("ts_second"),
        F.hour("join_time_ts").alias("time_hour"),
        F.minute("join_time_ts").alias("time_minute"),
        F.second("join_time_ts").alias("time_second"),
        F.date_add("join_date", 7).alias("plus_7_days"),
        F.date_sub("join_date", 7).alias("minus_7_days"),
        F.add_months("join_date", 1).alias("plus_1_month"),
        F.last_day("join_date").alias("last_day"),
        F.quarter("join_date").alias("quarter"),
        F.date_format("join_timestamp", "EEEE").alias("day_name"),
        F.date_format("join_timestamp", "dd-MM-yyyy HH:mm:ss").alias("formatted_ts"),
        F.datediff(F.current_date(), F.col("join_date")).alias("days_since_join")
    )

display(demo_datetime)

## String Function

Upper, lower, length, initcap

In [0]:
demo_string = df.select(
    "id",
    "name",
    F.upper("name").alias("upper_name"),
    F.lower("name").alias("lower_name"),
    F.initcap("name").alias("initcap_name"),
    F.length("name").alias("name_length")
)

display(demo_string)

Concatenate columns

In [0]:
demo_concat = df.select(
    "id",
    "name",
    "days_active",
    F.concat(F.col("name"), F.lit("_"), F.col("days_active")).alias("concat_col"),
    F.concat_ws("-", F.col("id"), F.col("name"), F.col("days_active")).alias("concat_ws_col")
)

display(demo_concat)

Substring and replace

In [0]:
demo_substring = df.select(
    "id",
    "name",
    F.substring("name", 1, 3).alias("first_3_chars"),
    F.regexp_replace("name", "a", "@").alias("replaced_name")
)

display(demo_substring)

Conditional functions

In [0]:
demo_when = df.select(
    "id",
    "name",
    "days_active",
    F.when(F.col("days_active") >= 180, "Highly Active")
     .when(F.col("days_active") >= 60, "Moderately Active")
     .otherwise("New")
     .alias("activity_status")
)

display(demo_when)

case when style with multiple conditions

In [0]:
demo_case = df.select(
    "id",
    "name",
    "join_timestamp",
    F.when(F.month("join_timestamp").isin(12, 1, 2), "Winter")
     .when(F.month("join_timestamp").isin(3, 4, 5), "Spring")
     .when(F.month("join_timestamp").isin(6, 7, 8), "Summer")
     .otherwise("Autumn")
     .alias("season")
)

display(demo_case)

## Null handling functions

In [0]:
df_null = df.unionByName(
    spark.createDataFrame(
        [(5, None, None, None, None)],
        df.schema
    )
)

display(df_null)

In [0]:
demo_null = df_null.select(
    "id",
    "name",
    "days_active",
    F.col("name").isNull().alias("name_is_null"),
    F.col("days_active").isNull().alias("days_active_is_null"),
    F.coalesce(F.col("name"), F.lit("Unknown")).alias("name_filled"),
    F.coalesce(F.col("days_active"), F.lit(0)).alias("days_active_filled")
)

display(demo_null)

In [0]:
df_null.fillna({
    "name": "Unknown",
    "days_active": 0,
    "join_timestamp": "1970-01-01" ,
    "join_time": "00:00:00"
}).display()

## Type Conversion

In [0]:
demo_cast = df.select(
    "id",
    "days_active",
    F.col("days_active").cast("string").alias("days_as_string"),
    F.col("id").cast("double").alias("id_as_double"),
    F.to_date("join_timestamp").alias("join_date")
)

display(demo_cast)

## Distinct, dropDuplicates and sort

Distinct and dropDuplicates

In [0]:
demo_distinct = df.select("name").distinct()
display(demo_distinct)

In [0]:
demo_dedup = df.dropDuplicates(["name"])
display(demo_dedup)

In [0]:
demo_sort = df.orderBy(F.col("days_active").asc())

display(demo_sort)

## Custom function

- Normal Python UDF
- Multi-column UDF
- SQL registered UDF
- Pandas UDF
- Struct-returning UDF

|Type|	Performance|	Best Use|
|--|--|--|
|Built-in Spark functions|	Best|	Always prefer when possible|
|Python UDF|	Slower|	Custom logic not available in Spark functions|
|Pandas UDF|	Better than Python UDF|	Batch-based custom processing|
|SQL UDF|	Convenient in SQL|	SQL-driven notebooks or pipelines|

Create demo dataframe

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime, time

data = [
    Row(id=1, name="Alice", days_active=120, join_timestamp=datetime(2024, 1, 15, 9, 30, 0), join_time=time(9, 30, 0)),
    Row(id=2, name="Bob", days_active=200, join_timestamp=datetime(2023, 12, 5, 14, 0, 0), join_time=time(14, 0, 0)),
    Row(id=3, name="Charlie", days_active=30, join_timestamp=datetime(2025, 6, 30, 8, 15, 0), join_time=time(8, 15, 0)),
    Row(id=4, name="Diana", days_active=60, join_timestamp=datetime(2026, 2, 28, 16, 45, 0), join_time=time(16, 45, 0))
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("days_active", IntegerType(), True),
    StructField("join_timestamp", TimestampType(), True),
    StructField("join_time", StringType(), True)
])

data = [
    Row(
        id=row.id,
        name=row.name,
        days_active=row.days_active,
        join_timestamp=row.join_timestamp,
        join_time=row.join_time.strftime("%H:%M:%S")
    )
    for row in data
]

df = spark.createDataFrame(data, schema)
display(df)

### Python UDF

In [0]:
def someFunction():
    print("Hello World")
    print("Another line")

print("Outside Function")
someFunction()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

def get_activity_status(days):
    if days is None:
        return "Unknown"
    elif days >= 180:
        return "Highly Active"
    elif days >= 60:
        return "Moderately Active"
    else:
        return "New"

activity_udf = udf(get_activity_status, StringType())  # extra step to use python function in spark code

df_activity = df.withColumn("activity_status", activity_udf(F.col("days_active")))

display(df_activity)

In [0]:
from pyspark.sql.types import StringType

def build_user_label(name, days):
    if name is None or days is None:
        return "Invalid"
    return f"{name}_active_{days}" #f string

label_udf = udf(build_user_label, StringType())

df_label = df.withColumn(
    "user_label",
    label_udf(F.col("name"), F.col("days_active"))
)

display(df_label)

### SQL UDF

In [0]:
from pyspark.sql.types import StringType

def custom_band(days):
    if days is None:
        return "Unknown"
    elif days >= 150:
        return "Gold"
    elif days >= 75:
        return "Silver"
    else:
        return "Bronze"

spark.udf.register("custom_band_udf", custom_band, StringType())

In [0]:
df.createOrReplaceTempView("users")

sql_result = spark.sql("""
    SELECT
        id,
        name,
        days_active,
        custom_band_udf(days_active) AS membership_band
    FROM users
""")

display(sql_result)

### Pandas UDF

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType

@pandas_udf(StringType())
def pandas_label_udf(name: pd.Series, days: pd.Series) -> pd.Series:
    return name.fillna("Unknown") + "_PD_" + days.fillna(0).astype(str)

df_pandas_udf = df.withColumn(
    "pandas_label",
    pandas_label_udf(F.col("name"), F.col("days_active"))
)

display(df_pandas_udf)

All at once

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType, ArrayType, StructType, StructField, IntegerType
import pandas as pd
from datetime import datetime

def get_activity_status(days):
    if days is None:
        return "Unknown"
    elif days >= 180:
        return "Highly Active"
    elif days >= 60:
        return "Moderately Active"
    else:
        return "New"

def get_time_slot(time_str):
    if time_str is None:
        return "Unknown"
    hour = int(time_str.split(":")[0])
    if hour < 12:
        return "Morning"
    elif hour < 17:
        return "Afternoon"
    else:
        return "Evening"

def classify_join_year(ts):
    if ts is None:
        return "Unknown"
    current_year = datetime.now().year
    if ts.year < current_year:
        return "Past Join"
    elif ts.year == current_year:
        return "Current Year Join"
    else:
        return "Future Join"

@pandas_udf(StringType())
def pandas_label_udf(name: pd.Series, days: pd.Series) -> pd.Series:
    return name.fillna("Unknown") + "_PD_" + days.fillna(0).astype(str)

activity_udf = udf(get_activity_status, StringType())
time_slot_udf = udf(get_time_slot, StringType())
join_year_udf = udf(classify_join_year, StringType())

demo_custom = df.withColumn("activity_status", activity_udf(F.col("days_active"))) \
    .withColumn("time_slot", time_slot_udf(F.col("join_time"))) \
    .withColumn("join_year_category", join_year_udf(F.col("join_timestamp"))) \
    .withColumn("pandas_label", pandas_label_udf(F.col("name"), F.col("days_active")))

display(demo_custom)